# Project 2 — Text Preprocessing with NLTK & spaCy

## Why preprocess?
Real-world text is **messy**: HTML tags, URLs, emoji, capitalization, punctuation, slang. 
Cleaning it before training a model:

- Reduces vocabulary size → faster training
- Removes noise → better accuracy
- Standardizes input → *Running, ran, runs* all become *run*

## The cleaning pipeline

```
Raw text
    │
    ├── lowercase
    ├── remove HTML / URLs / numbers
    ├── remove punctuation
    ├── tokenize
    ├── remove stop words
    ├── stem  (or)  lemmatize
    │
    ▼
  Clean tokens
```

## Stemming vs Lemmatization

| | Stemming | Lemmatization |
|---|---|---|
| Method | Chops off endings | Uses a dictionary |
| Speed | Very fast | Slower |
| Accuracy | Approximate (`amazingli`) | Exact (`amaze`) |
| Example | `running → run`, `studies → studi` | `running → run`, `studies → study` |

## Step 1 — Imports

In [1]:
import re
import string
import nltk
import spacy

for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet']:
    nltk.download(pkg, quiet=True)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

nlp = spacy.load('en_core_web_sm')

## Step 2 — A noisy sample text

This fake review contains *all the kinds of noise* you'll see in the wild.

In [2]:
raw_text = '''
<p>I LOVED this product!!! It's the BEST phone I have ever owned :-)
Visit https://example.com/review for more details. The battery is running
amazingly, and the cameras are better than my older phones. #Awesome 100%</p>
'''
print(raw_text)


<p>I LOVED this product!!! It's the BEST phone I have ever owned :-)
Visit https://example.com/review for more details. The battery is running
amazingly, and the cameras are better than my older phones. #Awesome 100%</p>



## Part A — Cleaning with NLTK

We'll write a single function that performs every cleaning step in order.

In [3]:
def preprocess_with_nltk(text):
    # 1. lowercase
    text = text.lower()
    # 2. remove HTML tags
    text = re.sub(r'<.*?>', ' ', text)
    # 3. remove URLs
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    # 4. remove digits
    text = re.sub(r'\d+', ' ', text)
    # 5. remove punctuation (! . , ? etc.)
    text = text.translate(str.maketrans('', '', string.punctuation))
    # 6. tokenize
    tokens = word_tokenize(text)
    # 7. remove stop words and very short tokens
    stop = set(stopwords.words('english'))
    tokens = [t for t in tokens if t not in stop and len(t) > 1]
    # 8a. stemming
    stemmer = PorterStemmer()
    stemmed = [stemmer.stem(t) for t in tokens]
    # 8b. lemmatization
    lemmatizer = WordNetLemmatizer()
    lemmatized = [lemmatizer.lemmatize(t, pos='v') for t in tokens]
    return tokens, stemmed, lemmatized

tokens, stemmed, lemmatized = preprocess_with_nltk(raw_text)
print('Cleaned tokens :', tokens)
print()
print('Stemmed        :', stemmed)
print()
print('Lemmatized     :', lemmatized)

Cleaned tokens : ['loved', 'product', 'best', 'phone', 'ever', 'owned', 'visit', 'details', 'battery', 'running', 'amazingly', 'cameras', 'better', 'older', 'phones', 'awesome']

Stemmed        : ['love', 'product', 'best', 'phone', 'ever', 'own', 'visit', 'detail', 'batteri', 'run', 'amazingli', 'camera', 'better', 'older', 'phone', 'awesom']

Lemmatized     : ['love', 'product', 'best', 'phone', 'ever', 'own', 'visit', 'detail', 'battery', 'run', 'amazingly', 'cameras', 'better', 'older', 'phone', 'awesome']


## Part B — Cleaning with spaCy

spaCy is more **elegant** because every token already has built-in attributes:
- `token.is_stop`  — *the, is, and…*
- `token.is_punct` — *!, ., ?…*
- `token.is_space` — whitespace tokens
- `token.lemma_`   — dictionary form (no extra library needed)

In [ ]:
def preprocess_with_spacy(text):
    text = text.lower()
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    doc = nlp(text)
    return [tok.lemma_ for tok in doc
            if not tok.is_stop and not tok.is_punct and not tok.is_space
            and len(tok.lemma_) > 1]

spacy_clean = preprocess_with_spacy(raw_text)
print(spacy_clean)

## Comparison

Quick side-by-side count to see how the pipelines differ.

In [ ]:
print(f'NLTK  produced {len(lemmatized)} tokens')
print(f'spaCy produced {len(spacy_clean)} tokens')

## Bonus — Batch preprocess a list of reviews

In a real project you'll loop over thousands of documents.

In [ ]:
reviews = [
    'I LOVE the new iPhone 15! Camera is amazing.',
    'Battery dies too fast. Not worth $999. Will return!!!',
    'Good build, decent screen, but software is buggy.',
]

for r in reviews:
    print(' -', preprocess_with_spacy(r))

## Summary

- Use **regex** to strip HTML, URLs, numbers.
- Use **stop-word removal** to drop low-information words.
- Use **lemmatization** (preferred) or **stemming** to normalize word forms.
- spaCy is more concise; NLTK gives you finer control.